In [ ]:
import sys
sys.path.append(r"C:\Users\yrro\OneDrive\PGE2 COURSES\ML\src")
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score, precision_score,
                             recall_score, f1_score, precision_recall_curve)
from credit_card_default.data import prepare_data
from credit_card_default.features import make_prep, add_features

# Load everything: df = the full cleaned table, then the train/test inputs (X) and answers (y)
df, X_train, X_test, y_train, y_test = prepare_data()
# Add 5 extra "engineered" columns (new columns computed from existing ones) to the training inputs
X_train_fe = add_features(X_train)                      # adds the 5 engineered columns
# Plan for fair testing: split the training data into 5 parts, keeping the same default share in each part
# (shuffle=True mixes the rows first; random_state=42 makes the shuffle repeatable)
cv = StratifiedKFold(5, shuffle=True, random_state=42)

# Define a function that builds our 3 models; class_weight says whether to give extra importance to defaulters
def base_models(class_weight=None):
    """Same settings as get_model in models.py; class_weight is what we vary here."""
    # Return a dictionary: each model's name -> the model itself
    return {
            # Logistic regression (max_iter=1000 = allowed to try up to 1000 times to learn)
            "logistic": LogisticRegression(class_weight=class_weight, max_iter=1000, random_state=42),
            # Linear SVM: draws a straight dividing line between defaulters and non-defaulters
            "linear SVM": LinearSVC(class_weight=class_weight, max_iter=5000, random_state=42),
            # k-NN: looks at the 50 most similar customers and goes with the majority (has no class_weight option)
            "k-NN (k=50)": KNeighborsClassifier(n_neighbors=50)}

ImportError: cannot import name 'add_features' from 'credit_card_default.features' (C:\Users\yrro\OneDrive\PGE2 COURSES\ML\src\credit_card_default\features.py)

In [ ]:
# Empty dictionary where we will store the results of every experiment
rows = {}
# Go through each of the 3 models (name + the model object)
for name, clf in base_models().items():
    # Try PAY columns as plain numbers (False), then as categories turned into 0/1 columns (True)
    for pay_cat in (False, True):
        # Try without (False), then with (True), the 5 engineered extra columns
        for eng in (False, True):
            # Build a pipeline: step 1 prepares the data, step 2 trains the model
            pipe = Pipeline([("prep", make_prep(pay_cat, eng)), ("model", clf)])
            # Test the pipeline 5 times with cross-validation, measuring AUC and AP (two ranking-quality scores)
            out = cross_validate(pipe, X_train_fe, y_train, cv=cv, scoring=["roc_auc", "average_precision"])
            # Save the results under a label like (model, "PAY one-hot"/"PAY numeric", "+ engineered"/"base"):
            # AUC = average over the 5 tests, AUC std = how much it varied between tests, AP = average AP
            rows[(name, "PAY one-hot" if pay_cat else "PAY numeric", "+ engineered" if eng else "base")] = {
                "AUC": out["test_roc_auc"].mean(), "AUC std": out["test_roc_auc"].std(),
                "AP": out["test_average_precision"].mean()}
# Turn the results into a table, flip it so each experiment is one row, and round to 4 decimals
pd.DataFrame(rows).T.round(4)

NameError: name 'base_models' is not defined

In [ ]:
# Define a function that tests one pipeline and returns a bunch of scores
def score_variant(pipe):
    # Grab the model itself: it is the last step of the pipeline
    model = pipe.steps[-1][1]
    # Use probabilities if the model can give them, otherwise use its raw "decision score" (LinearSVC can't give probabilities)
    method = "predict_proba" if hasattr(model, "predict_proba") else "decision_function"
    # For every training row, get a score from a model that never saw that row (cross-validation predictions)
    s = cross_val_predict(pipe, X_train_fe, y_train, cv=cv, method=method)
    # If we got probabilities, keep only the column for "will default" (class 1)
    if method == "predict_proba":
        s = s[:, 1]
    # The cutoff for saying "default": 0.5 for probabilities, 0.0 for decision scores
    thr = 0.5 if method == "predict_proba" else 0.0
    # Turn scores into hard predictions: 1 = default, 0 = no default
    pred = (s >= thr).astype(int)
    # Precision and recall at every possible cutoff
    # (precision = of those flagged, how many really defaulted; recall = of real defaulters, how many we caught)
    prec, rec, _ = precision_recall_curve(y_train, s)
    # F1 at every cutoff (a single score mixing precision and recall); np.clip avoids dividing by zero
    f1s = 2 * prec * rec / np.clip(prec + rec, 1e-12, None)
    # Return all the scores in one dictionary
    return {
            # AUC: how well the scores rank defaulters above non-defaulters (0.5 = guessing, 1 = perfect)
            "AUC": roc_auc_score(y_train, s),
            # AP: average precision, a ranking score that cares more about the (rarer) defaulters
            "AP": average_precision_score(y_train, s),
            # Recall at the default cutoff: share of real defaulters we caught
            "recall@default": recall_score(y_train, pred),
            # Precision at the default cutoff: share of our "default" flags that were right
            "precision@default": precision_score(y_train, pred, zero_division=0),
            # F1 at the default cutoff
            "F1@default": f1_score(y_train, pred),
            # The best F1 we could get if we picked the perfect cutoff
            "best F1": f1s.max(),
            # The best precision we can get while still catching at least 60% of defaulters
            "precision@recall>=.6": prec[rec >= 0.6].max(),
            # The average predicted probability (only for models that give probabilities; otherwise "not a number")
            "mean proba": s.mean() if method == "predict_proba" else np.nan}

# Empty dictionary to store the results (starting fresh)
rows = {}
# Two versions: no class weighting (None) and "balanced" weighting (defaulters count more during training)
for variant, cw in {"none": None, "class_weight": "balanced"}.items():
    # Go through each of the 3 models, built with this class_weight setting
    for name, clf in base_models(cw).items():
        # Skip k-NN when using class weights, because k-NN can't use them
        if variant == "class_weight" and name == "k-NN (k=50)":
            continue                                          # k-NN has no class_weight option
        # Test this model (with data prep including PAY one-hot and engineered columns) and store its scores
        rows[(name, variant)] = score_variant(Pipeline([("prep", make_prep(True, True)), ("model", clf)]))
# Turn results into a table, one row per experiment, sorted by name, rounded to 3 decimals
pd.DataFrame(rows).T.sort_index().round(3)

c:\Users\yrro\OneDrive\little yrrro\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [6, 7, 8] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\yrro\OneDrive\little yrrro\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [4] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\yrro\OneDrive\little yrrro\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [6, 7, 8] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\yrro\OneDrive\little yrrro\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [4] during transform. These unknown categories will be encoded

AUC     AP  recall@0.5  precision@0.5  F1@0.5  \
logistic      SMOTE         0.772  0.538       0.610          0.472   0.532   
              class_weight  0.772  0.539       0.614          0.472   0.534   
              none          0.772  0.541       0.359          0.680   0.470   
random forest SMOTE         0.779  0.546       0.534          0.525   0.530   
              class_weight  0.785  0.558       0.590          0.501   0.542   
              none          0.784  0.558       0.371          0.672   0.478   

                            best F1  mean proba  
logistic      SMOTE           0.537       0.432  
              class_weight    0.539       0.434  
              none            0.539       0.221  
random forest SMOTE           0.535       0.351  
              class_weight    0.545       0.382  
              none            0.545       0.224

In [ ]:
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# Go through each of the 3 models (with default settings)
for name, clf in base_models().items():
    # Build a pipeline: prepare data, then SMOTE (creates extra fake defaulter rows so both groups are balanced), then train
    pipe = ImbPipeline([("prep", make_prep(True, True)), ("smote", SMOTE(random_state=42)), ("model", clf)])
    # Test it and add its scores to the same results dictionary, labelled "SMOTE"
    rows[(name, "SMOTE")] = score_variant(pipe)
# Show all results (previous experiments + SMOTE) as one table, sorted and rounded to 3 decimals
pd.DataFrame(rows).T.sort_index().round(3)